# 01 · Chat Completions — the kitchen-sink tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/01-chat-completions.ipynb)

Everything you can do with `/chat/completions` on Venice. By the end you'll know how to:

1. Stream responses
2. Use system prompts (and disable Venice's default one)
3. Use `venice_parameters` for web search, web scraping, and characters
4. Call reasoning models with `reasoning_effort`
5. Get structured JSON responses with a schema
6. Multimodal input (vision)
7. Tool calling

Same SDK as OpenAI — every trick you already know works.

In [ ]:
%pip install --quiet openai requests python-dotenv rich

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. Streaming

`stream=True` returns chunks as they're generated.

In [ ]:
stream = client.chat.completions.create(
    model="zai-org-glm-4.7",
    messages=[{"role": "user", "content": "Write a haiku about Base mainnet."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## 2. System prompts

By default Venice prepends a system prompt that encourages uncensored, natural responses. To disable it pass `venice_parameters.include_venice_system_prompt = False`.

In [ ]:
resp = client.chat.completions.create(
    model="venice-uncensored",
    messages=[
        {"role": "system", "content": "You are a Venetian gondolier in 1530. Reply in character."},
        {"role": "user", "content": "Tell me about your day."},
    ],
    extra_body={
        "venice_parameters": {
            "include_venice_system_prompt": False,
        }
    },
)
print(resp.choices[0].message.content)

## 3. Web search — built in, no scaffolding

Pass `venice_parameters.enable_web_search = "auto"` (or `"on"`) and Venice does the search + grounding for you. Citations are returned in the response.

In [ ]:
resp = client.chat.completions.create(
    model="zai-org-glm-4.7",
    messages=[{"role": "user", "content": "What were the top 3 stories on Hacker News today?"}],
    extra_body={
        "venice_parameters": {
            "enable_web_search": "on",
            "enable_web_citations": True,
        }
    },
)
print(resp.choices[0].message.content)

### Same thing, but with a model suffix

For convenience you can also enable web search by appending `:web` to the model id.

In [ ]:
resp = client.chat.completions.create(
    model="zai-org-glm-4.7:web",
    messages=[{"role": "user", "content": "Latest Base ecosystem news headlines?"}],
)
print(resp.choices[0].message.content)

## 4. Reasoning models

Reasoning models (Qwen3, GLM-4.6 reasoning, etc.) think out loud before answering. Use `reasoning_effort` to control how much thinking they do (`low` / `medium` / `high`).

In [ ]:
resp = client.chat.completions.create(
    model="qwen3-235b",
    messages=[{
        "role": "user",
        "content": "If I stake 100 DIEM at $1/day each, what's my annual API credit value? Show your work.",
    }],
    extra_body={"reasoning_effort": "medium"},
)
print(resp.choices[0].message.content)

## 5. Structured outputs (JSON schema)

Pass a JSON schema in `response_format` and Venice will guarantee the model returns JSON matching it.

In [ ]:
import json

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "founded": {"type": "integer"},
        "famous_for": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["name", "founded", "famous_for"],
}

resp = client.chat.completions.create(
    model="zai-org-glm-4.7",
    messages=[{"role": "user", "content": "Tell me about Venice (the city)."}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "city_facts", "schema": schema, "strict": True},
    },
)

data = json.loads(resp.choices[0].message.content)
print(json.dumps(data, indent=2))

## 6. Vision — pass an image

Multimodal models accept images as URLs or base64.

In [ ]:
resp = client.chat.completions.create(
    model="mistral-31-24b",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What's in this image?"},
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/Venezia_-_Canal_Grande.jpg/640px-Venezia_-_Canal_Grande.jpg"
                },
            },
        ],
    }],
)
print(resp.choices[0].message.content)

## 7. Tool calling

Standard OpenAI tool-calling format. Useful for agents.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_usdc_price",
        "description": "Get the current USDC price (always 1.00 USD).",
        "parameters": {
            "type": "object",
            "properties": {"vs": {"type": "string"}},
            "required": ["vs"],
        },
    },
}]

resp = client.chat.completions.create(
    model="zai-org-glm-4.7",
    messages=[{"role": "user", "content": "What's USDC trading at vs USD?"}],
    tools=tools,
    tool_choice="auto",
)

msg = resp.choices[0].message
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"Model wants to call: {tc.function.name}({tc.function.arguments})")
else:
    print(msg.content)

## What just happened

You used the same `openai` Python SDK to:
- Stream from one model
- Roleplay with another
- Pull live web results without writing a single scraping line
- Get a reasoning model to show its work
- Get guaranteed JSON
- Describe an image
- Trigger a function call

Every Venice text model speaks the same protocol. Switching providers = changing one string.

**Next:** [02 · Embeddings + RAG](./02-embeddings-and-rag.ipynb)